## 🐴 Horse Survival Prediction

Given *medical data about horses*, let's try to predict whether a given horse will **survive** or not.

We will use a decision tree classifier and a random forest classifier to make our predictions.

Data source: https://www.kaggle.com/datasets/uciml/horse-colic

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
data = pd.read_csv('archive/horse.csv')
data

,surgery,age,hospital_number,rectal_temp,pulse,respiratory_rate,temp_of_extremities,peripheral_pulse,mucous_membrane,capillary_refill_time,...,packed_cell_volume,total_protein,abdomo_appearance,abdomo_protein,outcome,surgical_lesion,lesion_1,lesion_2,lesion_3,cp_data
0,no,adult,530101,38.5,66.0,28.0,cool,reduced,NaN,more_3_sec,...,45.0,8.4,NaN,NaN,died,no,11300,0,0,no
1,yes,adult,534817,39.2,88.0,20.0,NaN,NaN,pale_cyanotic,less_3_sec,...,50.0,85.0,cloudy,2.0,euthanized,no,2208,0,0,no
2,no,adult,530334,38.3,40.0,24.0,normal,normal,pale_pink,less_3_sec,...,33.0,6.7,NaN,NaN,lived,no,0,0,0,yes
3,yes,young,5290409,39.1,164.0,84.0,cold,normal,dark_cyanotic,more_3_sec,...,48.0,7.2,serosanguious,5.3,died,yes,2208,0,0,yes
4,no,adult,530255,37.3,104.0,35.0,NaN,NaN,dark_cyanotic,more_3_sec,...,74.0,7.4,NaN,NaN,died,no,4300,0,0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,yes,adult,533886,NaN,120.0,70.0,cold,NaN,pale_cyanotic,more_3_sec,...,55.0,65.0,NaN,NaN,euthanized,no,3205,0,0,no
295,no,adult,527702,37.2,72.0,24.0,cool,increased,pale_cyanotic,more_3_sec,...,44.0,NaN,serosanguious,3.3,euthanized,yes,2208,0,0,yes
296,yes,adult,529386,37.5,72.0,30.0,cold,reduced,pale_cyanotic,less_3_sec,...,60.0,6.8,NaN,NaN,died,yes,3205,0,0,no
297,yes,adult,530612,36.5,100.0,24.0,cool,reduced,pale_pink,less_3_sec,...,50.0,6.0,serosanguious,3.4,lived,yes,2208,0,0,yes


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   surgery                299 non-null    str    
 1   age                    299 non-null    str    
 2   hospital_number        299 non-null    int64  
 3   rectal_temp            239 non-null    float64
 4   pulse                  275 non-null    float64
 5   respiratory_rate       241 non-null    float64
 6   temp_of_extremities    243 non-null    str    
 7   peripheral_pulse       230 non-null    str    
 8   mucous_membrane        252 non-null    str    
 9   capillary_refill_time  267 non-null    str    
 10  pain                   244 non-null    str    
 11  peristalsis            255 non-null    str    
 12  abdominal_distention   243 non-null    str    
 13  nasogastric_tube       195 non-null    str    
 14  nasogastric_reflux     193 non-null    str    
 15  nasogastric_reflu

### Preprocessing

In [4]:
df = data.copy()

In [5]:
{column: list(df[column].unique()) for column in df.select_dtypes('str').columns}

{'surgery': ['no', 'yes'],
 'age': ['adult', 'young'],
 'temp_of_extremities': ['cool', nan, 'normal', 'cold', 'warm'],
 'peripheral_pulse': ['reduced', nan, 'normal', 'absent', 'increased'],
 'mucous_membrane': [nan,
  'pale_cyanotic',
  'pale_pink',
  'dark_cyanotic',
  'normal_pink',
  'bright_red',
  'bright_pink'],
 'capillary_refill_time': ['more_3_sec', 'less_3_sec', nan, '3'],
 'pain': ['extreme_pain',
  'mild_pain',
  'depressed',
  nan,
  'severe_pain',
  'alert'],
 'peristalsis': ['absent', 'hypomotile', nan, 'hypermotile', 'normal'],
 'abdominal_distention': ['severe', 'slight', 'none', nan, 'moderate'],
 'nasogastric_tube': [nan, 'none', 'slight', 'significant'],
 'nasogastric_reflux': [nan, 'less_1_liter', 'none', 'more_1_liter'],
 'rectal_exam_feces': ['decreased', 'absent', 'normal', nan, 'increased'],
 'abdomen': ['distend_large', 'other', 'normal', nan, 'firm', 'distend_small'],
 'abdomo_appearance': [nan, 'cloudy', 'serosanguious', 'clear'],
 'outcome': ['died', 'eut

In [6]:
binary_features = [
    'surgery',
    'age',
    'surgical_lesion',
    'cp_data'
]

positive_values = [
    'yes',
    'adult',
    'yes',
    'yes'
]


ordinal_features = [
    'temp_of_extremities',
    'peripheral_pulse',
    'capillary_refill_time',
    'pain',
    'peristalsis',
    'abdominal_distention',
    'nasogastric_tube',
    'nasogastric_reflux',
    'rectal_exam_feces'
]

orderings = [
    ['cold', 'cool', 'normal', 'warm'],
    ['absent', 'reduced', 'normal', 'increased'],
    ['less_3_sec', '3', 'more_3_sec'],
    ['alert', 'depressed', 'mild_pain', 'severe_pain', 'extreme_pain'],
    ['absent', 'hypomotile', 'normal', 'hypermotile'],
    ['none', 'slight', 'moderate', 'severe'],
    ['none', 'slight', 'significant'],
    ['none', 'less_1_liter', 'more_1_liter'],
    ['absent', 'decreased', 'normal', 'increased']
]


nominal_features = [
    'mucous_membrane',
    'abdomen',
    'abdomo_appearance',
    'hospital_number'
]

prefixes = [
    'MM',
    'AB',
    'AA',
    'HN'
]

In [7]:
def binary_encode(df, columns, positive_values):
    df = df.copy()
    for column, positive_value in zip(columns, positive_values):
        df[column] = df[column].apply(lambda x: 1 if x == positive_value else 0)
    return df

def ordinal_encode(df, columns, orderings):
    df = df.copy()
    for column, ordering in zip(columns, orderings):
        df[column] = df[column].apply(lambda x: ordering.index(x))
    return df

def onehot_encode(df, columns, prefixes):
    df = df.copy()
    for column, prefix in zip(columns, prefixes):
        dummies = pd.get_dummies(df[column], prefix=prefix, dtype=int)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(column, axis=1)
    return df

In [8]:
# Missing Values
df.isna().sum()

surgery                    0
age                        0
hospital_number            0
rectal_temp               60
pulse                     24
respiratory_rate          58
temp_of_extremities       56
peripheral_pulse          69
mucous_membrane           47
capillary_refill_time     32
pain                      55
peristalsis               44
abdominal_distention      56
nasogastric_tube         104
nasogastric_reflux       106
nasogastric_reflux_ph    246
rectal_exam_feces        102
abdomen                  118
packed_cell_volume        29
total_protein             33
abdomo_appearance        165
abdomo_protein           198
outcome                    0
surgical_lesion            0
lesion_1                   0
lesion_2                   0
lesion_3                   0
cp_data                    0
dtype: int64

In [9]:
for column in df.columns:
    if column in df.select_dtypes('str').columns:
        if column not in nominal_features:
            df[column] = df[column].fillna(df[column].mode()[0])
    else:
        df[column] = df[column].fillna(df[column].mean())

In [10]:
df.isna().sum()

surgery                    0
age                        0
hospital_number            0
rectal_temp                0
pulse                      0
respiratory_rate           0
temp_of_extremities        0
peripheral_pulse           0
mucous_membrane           47
capillary_refill_time      0
pain                       0
peristalsis                0
abdominal_distention       0
nasogastric_tube           0
nasogastric_reflux         0
nasogastric_reflux_ph      0
rectal_exam_feces          0
abdomen                  118
packed_cell_volume         0
total_protein              0
abdomo_appearance        165
abdomo_protein             0
outcome                    0
surgical_lesion            0
lesion_1                   0
lesion_2                   0
lesion_3                   0
cp_data                    0
dtype: int64

#### Encoding

In [11]:
# Binary encoding
df = binary_encode(df, binary_features, positive_values)

# Ordinal encoding
df = ordinal_encode(df, ordinal_features, orderings)

# OneHot encoding
df = onehot_encode(df, nominal_features, prefixes)

In [12]:
df

,surgery,age,rectal_temp,pulse,respiratory_rate,temp_of_extremities,peripheral_pulse,capillary_refill_time,pain,peristalsis,...,HN_5294369,HN_5294539,HN_5297159,HN_5297379,HN_5299253,HN_5299603,HN_5299629,HN_5301219,HN_5305129,HN_5305629
0,0,1,38.500000,66.0,28.0,1,1,2,4,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,39.200000,88.0,20.0,1,2,0,2,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,38.300000,40.0,24.0,2,2,0,2,1,...,0,0,0,0,0,0,0,0,0,0
3,1,0,39.100000,164.0,84.0,0,2,2,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0,1,37.300000,104.0,35.0,1,2,2,2,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,1,1,38.168619,120.0,70.0,0,2,2,1,0,...,0,0,0,0,0,0,0,0,0,0
295,0,1,37.200000,72.0,24.0,1,3,2,3,1,...,0,0,0,0,0,0,0,0,0,0
296,1,1,37.500000,72.0,30.0,0,1,0,3,0,...,0,0,0,0,0,0,0,0,0,0
297,1,1,36.500000,100.0,24.0,1,1,0,2,1,...,0,0,0,0,0,0,0,0,0,0


In [22]:
# Split df into X and y
y = df['outcome'].copy()
X = df.drop('outcome', axis=1).copy()

In [23]:
y

0            died
1      euthanized
2           lived
3            died
4            died
          ...    
294    euthanized
295    euthanized
296          died
297         lived
298    euthanized
Name: outcome, Length: 299, dtype: str

In [24]:
y.value_counts()

outcome
lived         178
died           77
euthanized     44
Name: count, dtype: int64

In [25]:
# Encode labels
# label_mapping = {'lived': 0, 'died': 1, 'euthanized': 2}

# y = y.replace(label_mapping)

y

0            died
1      euthanized
2           lived
3            died
4            died
          ...    
294    euthanized
295    euthanized
296          died
297         lived
298    euthanized
Name: outcome, Length: 299, dtype: str

In [26]:
# Scale X with a standard scaler
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [27]:
X

,surgery,age,rectal_temp,pulse,respiratory_rate,temp_of_extremities,peripheral_pulse,capillary_refill_time,pain,peristalsis,...,HN_5294369,HN_5294539,HN_5297159,HN_5297379,HN_5299253,HN_5299603,HN_5299629,HN_5301219,HN_5305129,HN_5305629
0,-1.229880,0.295420,0.506209,-0.218798,-0.155463,-0.473504,-1.088914,1.675999,1.731538,-1.185885,...,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928
1,0.813087,0.295420,1.575511,0.583463,-0.660914,-0.473504,0.671006,-0.601836,0.036841,-1.185885,...,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928
2,-1.229880,0.295420,0.200694,-1.166925,-0.408189,0.801970,0.671006,-0.601836,0.036841,-0.077824,...,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928
3,0.813087,-3.385016,1.422753,3.354910,3.382695,-1.748977,0.671006,1.675999,-0.810507,-1.185885,...,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928
4,-1.229880,0.295420,-1.326880,1.166925,0.286807,-0.473504,0.671006,1.675999,0.036841,-0.077824,...,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,0.813087,0.295420,0.000000,1.750388,2.498156,-1.748977,0.671006,1.675999,-0.810507,-1.185885,...,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928
295,-1.229880,0.295420,-1.479638,0.000000,-0.408189,-0.473504,2.430926,1.675999,0.884190,-0.077824,...,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928
296,0.813087,0.295420,-1.021366,0.000000,-0.029100,-1.748977,-1.088914,-0.601836,0.884190,-1.185885,...,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928
297,0.813087,0.295420,-2.548940,1.021059,-0.408189,-0.473504,-1.088914,-0.601836,0.036841,-0.077824,...,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928,-0.057928


### Training

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=123)

In [29]:
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

print("Decision Tree Accuracy: {:.2f}%".format(model.score(X_test, y_test)*100))

Decision Tree Accuracy: 65.56%


In [30]:
ensemble_model = RandomForestClassifier()
ensemble_model.fit(X_train, y_train)

print("Random Forest Accuracy: {:.2f}%".format(ensemble_model.score(X_test, y_test)*100))

Random Forest Accuracy: 73.33%
